# LeetCode 553 — Optimal Division bằng Python

Link đề: https://leetcode.com/problems/optimal-division/ · Độ khó: **Medium** · Chủ đề: Array, Math, Dynamic Programming

Skeleton LeetCode:

```python
class Solution:
    def optimalDivision(self, nums: List[int]) -> str:
        ...
```

Ghi chú: đề đảm bảo **chỉ có một phép chia tối ưu** (không chứa dấu ngoặc thừa), nên đáp án là **một chuỗi duy nhất**. Dù vậy, trong notebook ta vẫn test bằng cách **so sánh theo giá trị** (tính biểu thức bằng phân số chính xác) và **kiểm tra định dạng** (đúng thứ tự số, không ngoặc thừa), chứ không chỉ so chuỗi. Các ví dụ số 3, 4 là do notebook tự đặt, đáp án đã kiểm chứng bằng code; nếu cần hãy đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho dãy số nguyên `nums`, các số kề nhau được nối bằng phép **chia thực**; ta được thêm **dấu ngoặc** ở bất kỳ đâu để đổi thứ tự tính, sao cho giá trị biểu thức **lớn nhất**, rồi trả về biểu thức dạng chuỗi **không có ngoặc thừa**.
2. Nắm các công cụ Python: chia thực `/` so với chia nguyên `//`, `fractions.Fraction` để tính chính xác, `str()` + `"/".join(map(str, ...))`, f-string, cắt lát `nums[1:]`, đệ quy sinh mọi cách đặt ngoặc, DP theo đoạn.
3. Đi từ **vét cạn mọi cách đặt ngoặc** (số Catalan) → **DP theo đoạn lưu cả max lẫn min** (`O(n³)`) → **chỉ thử một cặp ngoặc** (`O(n³)` nhưng đơn giản) → **công thức toán `a0/(a1/a2/.../an-1)`** (`O(n)`).
4. Hiểu mấu chốt: mỗi biểu thức là tích `a_i^(±1)`, trong đó `a0` luôn là **tử**, `a1` luôn là **mẫu**, các số còn lại tuỳ ý; để lớn nhất ta đẩy **tất cả số còn lại lên tử** bằng đúng một cặp ngoặc bao từ `a1` đến hết.
5. Tránh các lỗi kinh điển: ngoặc thừa ở trường hợp `n = 2`, sai `n = 1`, bọc nhầm đầu biểu thức, quên `str()` khi nối chuỗi, và so sánh bằng số thực.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`. Các số **kề nhau** sẽ thực hiện **phép chia thực**. Ví dụ `nums = [2,3,4]` được hiểu là biểu thức `"2/3/4"`.

Tuy nhiên bạn có thể **thêm dấu ngoặc** vào bất cứ vị trí nào để thay đổi thứ tự thực hiện phép chia. Hãy thêm ngoặc sao cho giá trị biểu thức sau khi tính là **lớn nhất**, và trả về **biểu thức tương ứng dạng chuỗi**.

**Lưu ý:** biểu thức **không được chứa dấu ngoặc thừa** (dấu ngoặc không làm đổi thứ tự thực hiện phép toán).

**Ví dụ 1**

- Input: `nums = [1000,100,10,2]`
- Output: `"1000/(100/10/2)"`
- Giải thích: `1000/(100/10/2) = 1000/((100/10)/2) = 200`. Tuy nhiên cặp ngoặc đậm trong `"1000/((100/10)/2)"` là **thừa** vì không ảnh hưởng thứ tự tính, nên đáp án là `"1000/(100/10/2)"`. Các cách khác cho giá trị nhỏ hơn: `1000/(100/10)/2 = 50`, `1000/(100/(10/2)) = 50`, `1000/100/10/2 = 0.5`, `1000/100/(10/2) = 2`.

**Ví dụ 2**

- Input: `nums = [2,3,4]`
- Output: `"2/(3/4)"`
- Giải thích: `2/(3/4) = 8/3 ≈ 2.667`; thử mọi cách thì không có biểu thức nào lớn hơn.

**Ví dụ 3 (tự đặt, đối chiếu lại đề gốc nếu cần)**

- Input: `nums = [7]`
- Output: `"7"`

**Ví dụ 4 (tự đặt, đối chiếu lại đề gốc nếu cần)**

- Input: `nums = [7,3]`
- Output: `"7/3"` (chỉ có một cách, **không** viết `"7/(3)"`).

**Ràng buộc:**

- `1 <= nums.length <= 10`
- `2 <= nums[i] <= 1000`
- Với mỗi input, **chỉ có một** phép chia tối ưu.

Chú ý: vì chỉ có một phép chia tối ưu, đáp án là duy nhất, nhưng test vẫn nên **tính giá trị** và **kiểm tra không có ngoặc thừa** để chắc ta hiểu đúng đề.

In [1]:
from typing import List
from fractions import Fraction
from itertools import product
import random
import time

examples = [
    ([1000, 100, 10, 2], "1000/(100/10/2)"),
    ([2, 3, 4], "2/(3/4)"),
    ([7], "7"),                    # ví dụ tự đặt
    ([7, 3], "7/3"),               # ví dụ tự đặt
]
for nums, expected in examples:
    print(nums, "->", expected)

[1000, 100, 10, 2] -> 1000/(100/10/2)
[2, 3, 4] -> 2/(3/4)
[7] -> 7
[7, 3] -> 7/3


## 3. Kiến thức nền

### 3.1. Chia thực `/` và chia nguyên `//`

Trong Python 3, `a / b` luôn là **chia thực** (kết quả `float`), còn `a // b` là chia **làm tròn xuống** về số nguyên. Đề bài dùng chia **thực**, nên `100/10/2` là `5.0` chứ không phải `5`, và `1000/100/10/2` là `0.5` chứ không phải `0`.

In [2]:
print(7 / 2, 7 // 2)
print(100 / 10 / 2)
print(1000 / 100 / 10 / 2)       # 0.5
print(1000 // 100 // 10 // 2)    # 0  (không phải cái đề muốn)

3.5 3
5.0
0.5
0


### 3.2. `Fraction`: tính phân số chính xác

Số thực `float` có sai số làm tròn (ví dụ `0.1 + 0.2 != 0.3`). Khi cần **so sánh hai giá trị biểu thức** một cách đáng tin, hãy dùng `fractions.Fraction`: nó lưu tử số và mẫu số nguyên, nên cộng, trừ, nhân, chia **chính xác tuyệt đối**.

In [3]:
print(0.1 + 0.2 == 0.3)                                  # False (sai số)
print(Fraction(1, 10) + Fraction(2, 10) == Fraction(3, 10))   # True
x = Fraction(2) / (Fraction(3) / Fraction(4))
print(x, float(x))               # 8/3  2.666...
print(Fraction(1000) / Fraction(100) / Fraction(10) / Fraction(2))   # 1/2
print(Fraction(8, 3) > Fraction(5, 2))

False
True
8/3 2.6666666666666665
1/2
True


### 3.3. Nối chuỗi: `str()`, `join`, `map`, cắt lát và f-string

- `str(1000)` đổi số thành chuỗi `"1000"`.
- `"/".join(list_of_strings)` nối các chuỗi bằng dấu `/`. **Chú ý:** `join` chỉ nhận list **chuỗi**; muốn nối số phải đổi trước, thường bằng `map(str, nums)`.
- `nums[1:]` là lát cắt: mọi phần tử **từ vị trí 1 đến hết**.
- f-string: `f"{x}/({y})"` chèn giá trị biến vào chuỗi; dấu ngoặc tròn ngoài `{}` chỉ là ký tự thường.

In [4]:
nums = [1000, 100, 10, 2]
print("/".join(map(str, nums)))          # 1000/100/10/2
print(nums[1:])                          # [100, 10, 2]
print("/".join(map(str, nums[1:])))      # 100/10/2
print(f"{nums[0]}/({'/'.join(map(str, nums[1:]))})")   # 1000/(100/10/2)
try:
    "/".join(nums)                       # quên đổi sang chuỗi
except TypeError as e:
    print("TypeError:", e)

1000/100/10/2
[100, 10, 2]
100/10/2
1000/(100/10/2)
TypeError: sequence item 0: expected str instance, int found


### 3.4. Ngoặc đổi thứ tự tính: phép chia **kết hợp trái**

Không có ngoặc thì Python (và đề bài) tính **từ trái sang phải**: `a/b/c = (a/b)/c`. Ngoặc ở bên **phải** mới làm thay đổi kết quả: `a/(b/c) = a·c/b`. Nhớ hai điều sau, vì chúng quyết định mọi thứ trong bài này:

- `a/b/c = a / (b·c)` — b và c cùng ở **mẫu**.
- `a/(b/c) = a·c / b` — b ở **mẫu**, c bị đẩy lên **tử**.

In [5]:
a, b, c = Fraction(2), Fraction(3), Fraction(4)
print("a/b/c   =", a / b / c, " = a/(b*c) =", a / (b * c))
print("a/(b/c) =", a / (b / c), " = a*c/b   =", a * c / b)

a/b/c   = 1/6  = a/(b*c) = 1/6
a/(b/c) = 8/3  = a*c/b   = 8/3


### 3.5. Sinh mọi cách đặt ngoặc bằng đệ quy

Một cách đặt ngoặc chính là một cách **chia đoạn `nums[l..r]` thành hai phần** `nums[l..k]` và `nums[k+1..r]` rồi chia phần trái cho phần phải (như một cây nhị phân). Với 3 số có 2 cách: `a/b/c` và `a/(b/c)`. Với 4 số có 5 cách. Số cách cho `n` số là **số Catalan** `C(n−1)`: `1, 1, 2, 5, 14, 42, 132, 429, 1430, 4862` (với `n = 1..10`).

Quy ước viết chuỗi **không ngoặc thừa**: phần **trái** không bao giờ cần ngoặc (vì `(X)/Y = X/Y`), phần **phải** cần ngoặc **khi và chỉ khi** nó gồm từ hai số trở lên. Với quy ước này mỗi cách đặt ngoặc ứng với **đúng một** chuỗi chuẩn.

Ta cũng ghi lại **dấu mũ** của mỗi số: số nằm ở tử có mũ `+1`, số ở mẫu có mũ `−1`. Giá trị biểu thức là tích `a_i^(mũ_i)`.

In [6]:
def bracketings(nums, l, r, memo=None):
    """Mọi cách đặt ngoặc của nums[l..r]: list các (chuỗi chuẩn, tuple dấu mũ)."""
    if memo is None:
        memo = {}
    if (l, r) in memo:
        return memo[(l, r)]
    if l == r:
        res = [(str(nums[l]), (1,))]
    else:
        res = []
        for k in range(l, r):
            for ls, lsig in bracketings(nums, l, k, memo):
                for rs, rsig in bracketings(nums, k + 1, r, memo):
                    if k + 1 < r:                       # phần phải có >= 2 số: cần ngoặc
                        rs = "(" + rs + ")"
                    res.append((ls + "/" + rs, lsig + tuple(-s for s in rsig)))
    memo[(l, r)] = res
    return res

def signs_value(nums, signs):
    v = Fraction(1)
    for a, s in zip(nums, signs):
        v *= Fraction(a) ** s
    return v

for expr, sig in bracketings([2, 3, 4], 0, 2):
    print(expr, "dấu mũ", sig, "giá trị", signs_value([2, 3, 4], sig))
print([len(bracketings(list(range(2, 2 + n)), 0, n - 1)) for n in range(1, 11)])

2/(3/4) dấu mũ (1, -1, 1) giá trị 8/3
2/3/4 dấu mũ (1, -1, -1) giá trị 1/6
[1, 1, 2, 5, 14, 42, 132, 429, 1430, 4862]


### 3.6. Một "máy tính" nhỏ để kiểm tra đáp án (parser)

Để test chắc tay, ta viết một bộ phân tích cú pháp nhỏ: đọc chuỗi biểu thức, dựng thành cây (`num`, `div`, `paren`), **tính giá trị bằng `Fraction`** và **phát hiện ngoặc thừa**. Quy tắc ngoặc thừa (đúng với mọi biểu thức chỉ có phép chia):

- Ngoặc bọc **cả biểu thức** → thừa.
- Ngoặc bọc **toán hạng trái** của phép chia → thừa (`(X)/Y = X/Y`).
- Ngoặc bọc **một số đơn** hay bọc **ngoặc khác** → thừa.
- Ngoặc bọc **toán hạng phải** là một phép chia (`a/(b/c)`) → **cần thiết**.

In [7]:
def parse(expr):
    pos = 0

    def parse_expr():
        nonlocal pos
        node = parse_term()
        while pos < len(expr) and expr[pos] == "/":
            pos += 1
            node = ("div", node, parse_term())      # kết hợp trái
        return node

    def parse_term():
        nonlocal pos
        if pos < len(expr) and expr[pos] == "(":
            pos += 1
            inner = parse_expr()
            if pos >= len(expr) or expr[pos] != ")":
                raise ValueError("thiếu dấu )")
            pos += 1
            return ("paren", inner)
        start = pos
        while pos < len(expr) and expr[pos].isdigit():
            pos += 1
        if start == pos:
            raise ValueError("cần một số tại vị trí %d" % pos)
        return ("num", int(expr[start:pos]))

    node = parse_expr()
    if pos != len(expr):
        raise ValueError("còn ký tự thừa")
    return node

def eval_ast(node):
    if node[0] == "num":
        return Fraction(node[1])
    if node[0] == "paren":
        return eval_ast(node[1])
    return eval_ast(node[1]) / eval_ast(node[2])

def numbers_in_order(node):
    if node[0] == "num":
        return [node[1]]
    if node[0] == "paren":
        return numbers_in_order(node[1])
    return numbers_in_order(node[1]) + numbers_in_order(node[2])

def has_redundant(node, is_right=False, top=True):
    kind = node[0]
    if kind == "num":
        return False
    if kind == "paren":
        if top or not is_right:
            return True                  # bọc cả biểu thức hoặc toán hạng trái
        if node[1][0] != "div":
            return True                  # bọc số đơn hoặc bọc ngoặc khác
        return has_redundant(node[1], False, False)
    return has_redundant(node[1], False, False) or has_redundant(node[2], True, False)

def evaluate(expr):
    return eval_ast(parse(expr))

for e in ["1000/(100/10/2)", "1000/((100/10)/2)", "(1000/100)/10/2", "5/(2)", "2/(3/4)", "((2/3))"]:
    ast = parse(e)
    print(f"{e:22} giá trị {str(eval_ast(ast)):>8}  ngoặc thừa: {has_redundant(ast)}")

1000/(100/10/2)        giá trị      200  ngoặc thừa: False
1000/((100/10)/2)      giá trị      200  ngoặc thừa: True
(1000/100)/10/2        giá trị      1/2  ngoặc thừa: True
5/(2)                  giá trị      5/2  ngoặc thừa: True
2/(3/4)                giá trị      8/3  ngoặc thừa: False
((2/3))                giá trị      2/3  ngoặc thừa: True


### 3.7. "Nhiều đáp án đúng?" — đọc kỹ đề

Thoạt nhìn bài dễ có nhiều biểu thức cho cùng giá trị lớn nhất (ví dụ `1000/(100/10/2)` và `1000/((100/10)/2)` cho cùng `200`). Nhưng cái sau chứa **ngoặc thừa**, bị loại. Sau khi loại hết các cách viết có ngoặc thừa, đề **đảm bảo** chỉ còn **một** biểu thức đạt giá trị lớn nhất (ta sẽ tự kiểm chứng điều này bằng code ở mục 12).

Vì vậy với bài này: đáp án là duy nhất, có thể so sánh chuỗi; nhưng để chắc hơn, ta còn kiểm tra qua **giá trị** (`Fraction`) và **định dạng** (đúng thứ tự số, không ngoặc thừa) bằng bộ parser vừa viết.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Xét `nums = [2, 3, 4]`. Chỉ có hai cách đặt ngoặc:

- `2/3/4 = 2/(3·4) = 2/12 = 1/6`
- `2/(3/4) = 2·4/3 = 8/3`

Quan sát: trong cả hai, số `2` luôn ở **tử** (mũ `+1`), số `3` luôn ở **mẫu** (mũ `−1`), còn số `4` thì **tuỳ cách đặt ngoặc**: ở mẫu trong cách 1 (mũ `−1`), ở tử trong cách 2 (mũ `+1`). Cách nào lớn hơn? Cách đưa `4` lên tử.

Thử `nums = [1000, 100, 10, 2]` theo bảng dấu mũ (tử `+`, mẫu `−`):

| Biểu thức | mũ của `1000, 100, 10, 2` | Giá trị |
|---|---|---|
| `1000/100/10/2` | `+ − − −` | `1000/(100·10·2) = 0.5` |
| `1000/(100/10)/2` | `+ − + −` | `1000·10/(100·2) = 50` |
| `1000/(100/10/2)` | `+ − + +` | `1000·10·2/100 = 200` |

Càng nhiều số được đưa lên tử thì giá trị càng lớn. Câu hỏi: **tối đa có thể đưa bao nhiêu số lên tử?** Hai số đầu thì không thể: `a0` luôn ở tử, `a1` luôn ở mẫu. Vậy tốt nhất là `+ − + + + ... +`, và biểu thức `1000/(100/10/2)` đã đạt đúng dạng này.

In [8]:
nums = [1000, 100, 10, 2]
for expr, sig in bracketings(nums, 0, 3):
    print(f"{expr:20} mũ {sig}  giá trị {float(signs_value(nums, sig)):g}")

1000/(100/(10/2))    mũ (1, -1, 1, -1)  giá trị 50
1000/(100/10/2)      mũ (1, -1, 1, 1)  giá trị 200
1000/100/(10/2)      mũ (1, -1, -1, 1)  giá trị 2
1000/(100/10)/2      mũ (1, -1, 1, -1)  giá trị 50
1000/100/10/2        mũ (1, -1, -1, -1)  giá trị 0.5


## 5. Cách 1 — Brute Force / baseline: sinh mọi cách đặt ngoặc và chọn giá trị lớn nhất

**Ý tưởng "dịch nguyên văn đề bài":**

1. Dùng `bracketings` để sinh **mọi** cách đặt ngoặc (mỗi cách là một chuỗi chuẩn không ngoặc thừa, kèm dấu mũ).
2. Tính giá trị mỗi cách bằng `Fraction`.
3. Trả về chuỗi có giá trị **lớn nhất**.

Nhờ quy ước chuỗi chuẩn ở mục 3.5, mọi cách có ngoặc thừa đã tự bị loại, nên ta không cần lọc thêm.

In [9]:
def optimal_brute(nums):
    best_val, best_expr = None, None
    for expr, sig in bracketings(nums, 0, len(nums) - 1):
        v = signs_value(nums, sig)
        if best_val is None or v > best_val:
            best_val, best_expr = v, expr
    return best_expr

for nums, expected in examples:
    got = optimal_brute(nums)
    print(nums, "->", got, "| kỳ vọng", expected)
    assert got == expected

[1000, 100, 10, 2] -> 1000/(100/10/2) | kỳ vọng 1000/(100/10/2)
[2, 3, 4] -> 2/(3/4) | kỳ vọng 2/(3/4)
[7] -> 7 | kỳ vọng 7
[7, 3] -> 7/3 | kỳ vọng 7/3


**Độ phức tạp.** Số cách đặt ngoặc cho `n` số là số Catalan `C(n−1) ~ 4^n / n^1.5`: `n = 10` cho `4862` cách (nhỏ!), nhưng tốc độ tăng là **hàm mũ**, `n = 20` đã là `1.7 · 10^9`.

- **Time:** `O(C(n−1) · n)` ≈ mũ theo `n` (với `n ≤ 10` vẫn chạy được).
- **Space:** `O(C(n−1) · n)` cho danh sách các chuỗi (ta lưu cả danh sách theo từng đoạn).

**Nút thắt:** nếu `n` lớn thì không dùng được. Hơn nữa ta đang dò cả những cách đặt ngoặc **rõ ràng tệ**. Ta cần một cách **không phải duyệt hết**.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước 1 — tích của luỹ thừa `±1`.** Mọi biểu thức chỉ gồm phép chia (và ngoặc) có giá trị `a0^{s0} · a1^{s1} · ... · a_{n-1}^{s_{n-1}}` với mỗi `s_i ∈ {+1, −1}`.

**Bước 2 — hai dấu cố định.** Nếu `n ≥ 2`:

- `s0 = +1` **luôn luôn**: `a0` là số đầu tiên (ngoài cùng bên trái), nó nằm ở toán hạng **trái** của mọi phép chia chứa nó, nên luôn ở tử.
- `s1 = −1` **luôn luôn** (quy nạp theo `n`): với `n = 2` chỉ có `a0/a1`. Với `n ≥ 3`, xét phép chia ngoài cùng `X / Y`. Nếu `X` có từ hai số trở lên thì `a0, a1` thuộc `X`; theo giả thiết quy nạp `a1` có mũ `−1` trong `X`, và `X` là toán hạng trái nên mũ giữ nguyên. Nếu `X` chỉ là `a0` thì `a1` là **số đầu tiên của `Y`**; số đầu tiên của mọi biểu thức có mũ `+1` (đã chứng minh ở ý trên), rồi `Y` là toán hạng phải nên mũ bị lật thành `−1`.

**Bước 3 — cận trên.** Các số `a2..a_{n-1}` mỗi số có mũ `+1` hoặc `−1`. Vì `a_i ≥ 2 > 1`, mũ `+1` cho giá trị **lớn hơn** mũ `−1` (`a_i > 1/a_i`). Vậy giá trị lớn nhất có thể có là

```
a0 · a2 · a3 · ... · a_{n-1} / a1
```

**Bước 4 — đạt được cận trên.** Biểu thức `a0 / (a1 / a2 / ... / a_{n-1})` có đúng dấu mũ `+ − + + ... +` vì trong ngoặc `a1/a2/.../a_{n-1}` thì `a1` ở tử, các số còn lại ở mẫu, và cả ngoặc nằm ở mẫu của phép chia ngoài nên các mũ bị **lật dấu**: `a1` thành `−`, `a2..` thành `+`.

**Duy nhất:** vì `a_i > 1/a_i` **chặt** (do `a_i ≥ 2`), chỉ biểu thức có **mọi** `a2..a_{n-1}` ở tử mới đạt cận; và chỉ có một cách đặt ngoặc (không thừa) cho dấu mũ `+ − + + ... +`: đó chính là `a0/(a1/a2/.../a_{n-1})`.

Lưu ý riêng: với `n = 1` chỉ có `"a0"`, với `n = 2` chỉ có `"a0/a1"`, và **không** được viết `a0/(a1)` vì ngoặc thừa.

In [10]:
# Kiểm tra Bước 2 và 4 trực tiếp trên mọi cách đặt ngoặc của vài input
for nums in ([2, 3, 4], [1000, 100, 10, 2], [5, 7, 9, 11, 13]):
    all_sig = [sig for _, sig in bracketings(nums, 0, len(nums) - 1)]
    assert all(s[0] == +1 and s[1] == -1 for s in all_sig)
    best_sig = max(all_sig, key=lambda s: signs_value(nums, s))
    print(nums, "dấu mũ tốt nhất:", best_sig)
    assert best_sig == (1, -1) + (1,) * (len(nums) - 2)

[2, 3, 4] dấu mũ tốt nhất: (1, -1, 1)
[1000, 100, 10, 2] dấu mũ tốt nhất: (1, -1, 1, 1)
[5, 7, 9, 11, 13] dấu mũ tốt nhất: (1, -1, 1, 1, 1)


## 7. Các cách tiếp theo

### Cách 2 — DP theo đoạn: lưu cả giá trị **lớn nhất** và **nhỏ nhất** của mỗi đoạn (`O(n³)`)

Cách này **không cần biết** công thức toán, chỉ cần biết tính chất của phép chia số dương.

**Giải thích từng đoạn:**

1. Với đoạn `nums[l..r]`, mọi cách đặt ngoặc là `X / Y` với `X` là cách đặt ngoặc của `nums[l..k]`, `Y` của `nums[k+1..r]` (với một `k` nào đó).
2. Mọi số đều dương nên `X / Y` **lớn nhất** khi `X` **lớn nhất** và `Y` **nhỏ nhất**; `X / Y` **nhỏ nhất** khi `X` nhỏ nhất và `Y` lớn nhất. Hai đoạn `X`, `Y` không chung số nào nên chọn độc lập được.
3. Vì thế mỗi đoạn lưu hai thứ: `mx[l][r]` = (giá trị lớn nhất, chuỗi) và `mn[l][r]` = (giá trị nhỏ nhất, chuỗi).
4. Duyệt theo độ dài đoạn tăng dần (`length = 2..n`), thử mọi điểm cắt `k`:
   - `mx[l][r] = max_k ( mx[l][k].val / mn[k+1][r].val )`
   - `mn[l][r] = min_k ( mn[l][k].val / mx[k+1][r].val )`
5. Khi ghép chuỗi: phần trái viết nguyên, phần phải bọc ngoặc nếu có từ hai số trở lên.
6. Đáp án là chuỗi của `mx[0][n-1]`.

In [11]:
def optimal_dp(nums):
    n = len(nums)
    mx = [[None] * n for _ in range(n)]
    mn = [[None] * n for _ in range(n)]
    for i in range(n):
        mx[i][i] = mn[i][i] = (Fraction(nums[i]), str(nums[i]))

    for length in range(2, n + 1):
        for l in range(0, n - length + 1):
            r = l + length - 1
            best_max = best_min = None
            for k in range(l, r):
                # ứng viên cho max: (max trái) / (min phải)
                lv, ls = mx[l][k]
                rv, rs = mn[k + 1][r]
                if k + 1 < r:
                    rs = "(" + rs + ")"
                cand = (lv / rv, ls + "/" + rs)
                if best_max is None or cand[0] > best_max[0]:
                    best_max = cand
                # ứng viên cho min: (min trái) / (max phải)
                lv, ls = mn[l][k]
                rv, rs = mx[k + 1][r]
                if k + 1 < r:
                    rs = "(" + rs + ")"
                cand = (lv / rv, ls + "/" + rs)
                if best_min is None or cand[0] < best_min[0]:
                    best_min = cand
            mx[l][r], mn[l][r] = best_max, best_min
    return mx[0][n - 1][1]

for nums, expected in examples:
    got = optimal_dp(nums)
    print(nums, "->", got)
    assert got == expected

[1000, 100, 10, 2] -> 1000/(100/10/2)
[2, 3, 4] -> 2/(3/4)
[7] -> 7
[7, 3] -> 7/3


In [12]:
from fractions import Fraction


class Solution:
    def optimalDivision(self, nums: List[int]) -> str:
        n = len(nums)
        mx = [[None] * n for _ in range(n)]
        mn = [[None] * n for _ in range(n)]
        for i in range(n):
            mx[i][i] = mn[i][i] = (Fraction(nums[i]), str(nums[i]))
        for length in range(2, n + 1):
            for l in range(n - length + 1):
                r = l + length - 1
                best_max = best_min = None
                for k in range(l, r):
                    lv, ls = mx[l][k]
                    rv, rs = mn[k + 1][r]
                    if k + 1 < r:
                        rs = "(" + rs + ")"
                    cand = (lv / rv, ls + "/" + rs)
                    if best_max is None or cand[0] > best_max[0]:
                        best_max = cand
                    lv, ls = mn[l][k]
                    rv, rs = mx[k + 1][r]
                    if k + 1 < r:
                        rs = "(" + rs + ")"
                    cand = (lv / rv, ls + "/" + rs)
                    if best_min is None or cand[0] < best_min[0]:
                        best_min = cand
                mx[l][r], mn[l][r] = best_max, best_min
        return mx[0][n - 1][1]

print([Solution().optimalDivision(v) for v, _ in examples])

['1000/(100/10/2)', '2/(3/4)', '7', '7/3']


**Độ phức tạp (cách 2):**

- **Time:** `O(n³)` phép tính phân số: `O(n²)` đoạn × `O(n)` điểm cắt (cộng chi phí ghép chuỗi `O(n)` mỗi lần, tổng `O(n⁴)` ký tự nhưng với `n ≤ 10` là không đáng kể). Phân số `Fraction` có thể có tử và mẫu lớn, nhưng với `n ≤ 10` và `a_i ≤ 1000` vẫn nhỏ.
- **Space:** `O(n²)` ô, mỗi ô giữ một chuỗi dài `O(n)` → `O(n³)` ký tự.

Ưu điểm: dùng được cho bài toán tổng quát hơn (ví dụ các số có thể là phân số hay số lẻ mà công thức "một cặp ngoặc" không còn đúng).

### Cách 3 — Thu hẹp không gian tìm kiếm: chỉ thử **một** cặp ngoặc

**Ý tưởng:** Từ phân tích dấu mũ, ta biết lời giải tối ưu chỉ có tối đa **một cặp ngoặc** (bao từ `a1` tới hết). Nếu ta chưa chứng minh được điều đó, ta vẫn có thể **dùng nó như một giả thuyết** để giảm việc dò từ hàm mũ xuống đa thức: chỉ thử các biểu thức có **không hoặc một** cặp ngoặc.

**Giải thích từng đoạn:**

1. Một cặp ngoặc **không thừa** bao các số `nums[i..j]` phải có `i ≥ 1` (bao từ số đầu tiên thì thành thừa), `j > i` (bao một số đơn thì thừa).
2. Giá trị: tính nhóm `nums[i..j]` bằng chia trái-sang-phải thành một số `g`, rồi chia từ trái sang phải với các số còn lại.
3. Ứng viên "không ngoặc": `nums[0]/nums[1]/.../nums[n-1]`.
4. Lấy ứng viên có giá trị lớn nhất.

In [13]:
def optimal_one_pair(nums):
    n = len(nums)

    def value_with_group(i, j):
        """Giá trị khi bọc nums[i..j] bằng một cặp ngoặc (i, j = -1 nghĩa là không ngoặc)."""
        terms, t = [], 0
        while t < n:
            if t == i:
                g = Fraction(nums[i])
                for u in range(i + 1, j + 1):
                    g /= nums[u]
                terms.append(g)
                t = j + 1
            else:
                terms.append(Fraction(nums[t]))
                t += 1
        v = terms[0]
        for x in terms[1:]:
            v /= x
        return v

    def expr_with_group(i, j):
        parts = [str(x) for x in nums]
        if i >= 0:
            parts[i] = "(" + parts[i]
            parts[j] = parts[j] + ")"
        return "/".join(parts)

    best_val, best_expr = value_with_group(-1, -1), expr_with_group(-1, -1)
    for i in range(1, n):
        for j in range(i + 1, n):
            v = value_with_group(i, j)
            if v > best_val:
                best_val, best_expr = v, expr_with_group(i, j)
    return best_expr

for nums, expected in examples:
    got = optimal_one_pair(nums)
    print(nums, "->", got)
    assert got == expected

[1000, 100, 10, 2] -> 1000/(100/10/2)
[2, 3, 4] -> 2/(3/4)
[7] -> 7
[7, 3] -> 7/3


In [14]:
from fractions import Fraction


class Solution:
    def optimalDivision(self, nums: List[int]) -> str:
        n = len(nums)

        def value_with_group(i, j):
            terms, t = [], 0
            while t < n:
                if t == i:
                    g = Fraction(nums[i])
                    for u in range(i + 1, j + 1):
                        g /= nums[u]
                    terms.append(g)
                    t = j + 1
                else:
                    terms.append(Fraction(nums[t]))
                    t += 1
            v = terms[0]
            for x in terms[1:]:
                v /= x
            return v

        def expr_with_group(i, j):
            parts = [str(x) for x in nums]
            if i >= 0:
                parts[i] = "(" + parts[i]
                parts[j] = parts[j] + ")"
            return "/".join(parts)

        best_val, best_expr = value_with_group(-1, -1), expr_with_group(-1, -1)
        for i in range(1, n):
            for j in range(i + 1, n):
                v = value_with_group(i, j)
                if v > best_val:
                    best_val, best_expr = v, expr_with_group(i, j)
        return best_expr

print([Solution().optimalDivision(v) for v, _ in examples])

['1000/(100/10/2)', '2/(3/4)', '7', '7/3']


**Độ phức tạp (cách 3):**

- **Time:** `O(n²)` cặp `(i, j)` × `O(n)` tính giá trị = `O(n³)`.
- **Space:** `O(n)`.

Lưu ý quan trọng: cách này **dựa vào giả thuyết** "một cặp ngoặc là đủ". Ta sẽ đối chiếu nó với vét cạn để chắc chắn nó đúng trên mọi input nhỏ, và mục 10 chứng minh vì sao đúng.

### Cách 4 — Công thức toán `a0/(a1/a2/.../an-1)` (`O(n)`)

Từ phân tích ở mục 6: biểu thức tối ưu luôn là số đầu chia cho **một ngoặc bao toàn bộ phần còn lại** viết theo chia trái-sang-phải.

**Giải thích từng đoạn:**

1. `n == 1`: chỉ có một số, trả `str(nums[0])`.
2. `n == 2`: chỉ có `a0/a1`; thêm ngoặc sẽ thừa → trả `f"{nums[0]}/{nums[1]}"`.
3. `n >= 3`: trả `f"{nums[0]}/({'/'.join(map(str, nums[1:]))})"`.

In [15]:
def optimal_formula(nums):
    n = len(nums)
    if n == 1:
        return str(nums[0])
    if n == 2:
        return f"{nums[0]}/{nums[1]}"
    return f"{nums[0]}/({'/'.join(map(str, nums[1:]))})"

for nums, expected in examples:
    got = optimal_formula(nums)
    print(nums, "->", got)
    assert got == expected

[1000, 100, 10, 2] -> 1000/(100/10/2)
[2, 3, 4] -> 2/(3/4)
[7] -> 7
[7, 3] -> 7/3


In [16]:
class Solution:
    def optimalDivision(self, nums: List[int]) -> str:
        n = len(nums)
        if n == 1:
            return str(nums[0])
        if n == 2:
            return f"{nums[0]}/{nums[1]}"
        return f"{nums[0]}/({'/'.join(map(str, nums[1:]))})"

print([Solution().optimalDivision(v) for v, _ in examples])

['1000/(100/10/2)', '2/(3/4)', '7', '7/3']


**Độ phức tạp (cách 4):**

- **Time:** `O(n)` (xây chuỗi dài tỉ lệ với `n`).
- **Space:** `O(n)` cho chuỗi kết quả.

## 8. TRACE từng bước

### 8.1. Trace cách 2 (DP theo đoạn) trên `[1000, 100, 10, 2]`

Bảng giá trị `(max, min)` theo độ dài đoạn (chỉ số từ 0):

| Đoạn | Điểm cắt `k` và ứng viên max | `mx` | `mn` |
|---|---|---|---|
| `[100,10]` | `100/10` | `10` | `10` |
| `[1000,100]` | `1000/100` | `10` | `10` |
| `[10,2]` | `10/2` | `5` | `5` |
| `[1000,100,10]` | `k=0: 1000/(100/10) = 100`; `k=1: (1000/100)/10 = 1` | `100` (`1000/(100/10)`) | `1` (`1000/100/10`) |
| `[100,10,2]` | `k=0: 100/(10/2) = 20`; `k=1: (100/10)/2 = 5` | `20` (`100/(10/2)`) | `5` (`100/10/2`) |
| `[1000,100,10,2]` | `k=0: 1000/mn[1..3] = 1000/5 = 200`; `k=1: 10/mn[2..3] = 10/5 = 2`; `k=2: 100/2 = 50` | **`200`** (`1000/(100/10/2)`) | `0.5` |

Hàm `trace_dp` in đúng quá trình này (mỗi ô, mỗi điểm cắt).

In [17]:
def trace_dp(nums):
    n = len(nums)
    mx = [[None] * n for _ in range(n)]
    mn = [[None] * n for _ in range(n)]
    for i in range(n):
        mx[i][i] = mn[i][i] = (Fraction(nums[i]), str(nums[i]))
    for length in range(2, n + 1):
        print(f"--- độ dài đoạn {length} ---")
        for l in range(0, n - length + 1):
            r = l + length - 1
            best_max = best_min = None
            for k in range(l, r):
                lv, ls = mx[l][k]
                rv, rs = mn[k + 1][r]
                rs_p = "(" + rs + ")" if k + 1 < r else rs
                c_max = (lv / rv, ls + "/" + rs_p)
                lv2, ls2 = mn[l][k]
                rv2, rs2 = mx[k + 1][r]
                rs2_p = "(" + rs2 + ")" if k + 1 < r else rs2
                c_min = (lv2 / rv2, ls2 + "/" + rs2_p)
                print(f"  đoạn {nums[l:r + 1]} cắt sau vị trí {k}: max-ứng-viên {c_max[1]} = {c_max[0]}"
                      f" | min-ứng-viên {c_min[1]} = {c_min[0]}")
                if best_max is None or c_max[0] > best_max[0]:
                    best_max = c_max
                if best_min is None or c_min[0] < best_min[0]:
                    best_min = c_min
            mx[l][r], mn[l][r] = best_max, best_min
            print(f"  => mx = {best_max[0]} ({best_max[1]}) ; mn = {best_min[0]} ({best_min[1]})")
    return mx[0][n - 1][1]

print("kết quả:", trace_dp([1000, 100, 10, 2]))

--- độ dài đoạn 2 ---
  đoạn [1000, 100] cắt sau vị trí 0: max-ứng-viên 1000/100 = 10 | min-ứng-viên 1000/100 = 10
  => mx = 10 (1000/100) ; mn = 10 (1000/100)
  đoạn [100, 10] cắt sau vị trí 1: max-ứng-viên 100/10 = 10 | min-ứng-viên 100/10 = 10
  => mx = 10 (100/10) ; mn = 10 (100/10)
  đoạn [10, 2] cắt sau vị trí 2: max-ứng-viên 10/2 = 5 | min-ứng-viên 10/2 = 5
  => mx = 5 (10/2) ; mn = 5 (10/2)
--- độ dài đoạn 3 ---
  đoạn [1000, 100, 10] cắt sau vị trí 0: max-ứng-viên 1000/(100/10) = 100 | min-ứng-viên 1000/(100/10) = 100
  đoạn [1000, 100, 10] cắt sau vị trí 1: max-ứng-viên 1000/100/10 = 1 | min-ứng-viên 1000/100/10 = 1
  => mx = 100 (1000/(100/10)) ; mn = 1 (1000/100/10)
  đoạn [100, 10, 2] cắt sau vị trí 1: max-ứng-viên 100/(10/2) = 20 | min-ứng-viên 100/(10/2) = 20
  đoạn [100, 10, 2] cắt sau vị trí 2: max-ứng-viên 100/10/2 = 5 | min-ứng-viên 100/10/2 = 5
  => mx = 20 (100/(10/2)) ; mn = 5 (100/10/2)
--- độ dài đoạn 4 ---
  đoạn [1000, 100, 10, 2] cắt sau vị trí 0: max-ứng-viê

### 8.2. Trace cách 4 (công thức) trên vài input

Mỗi bước quyết định theo `n`, rồi dựng chuỗi từ các phần: `nums[0]`, phần còn lại `nums[1:]`, rồi ghép.

In [18]:
def trace_formula(nums):
    n = len(nums)
    print(f"nums = {nums}, n = {n}")
    if n == 1:
        print("  n == 1 -> chỉ một số, không có phép chia nào")
        return str(nums[0])
    if n == 2:
        print("  n == 2 -> chỉ có một cách, thêm ngoặc sẽ thừa")
        return f"{nums[0]}/{nums[1]}"
    tail = nums[1:]
    print("  phần số đầu    :", nums[0])
    print("  phần còn lại   :", tail)
    joined = "/".join(map(str, tail))
    print("  nối bằng '/'   :", joined)
    res = f"{nums[0]}/({joined})"
    print("  ghép với ngoặc :", res, "-> giá trị", evaluate(res))
    return res

for nums in ([7], [7, 3], [2, 3, 4], [1000, 100, 10, 2]):
    print("=>", trace_formula(nums))
    print()

nums = [7], n = 1
  n == 1 -> chỉ một số, không có phép chia nào
=> 7

nums = [7, 3], n = 2
  n == 2 -> chỉ có một cách, thêm ngoặc sẽ thừa
=> 7/3

nums = [2, 3, 4], n = 3
  phần số đầu    : 2
  phần còn lại   : [3, 4]
  nối bằng '/'   : 3/4
  ghép với ngoặc : 2/(3/4) -> giá trị 8/3
=> 2/(3/4)

nums = [1000, 100, 10, 2], n = 4
  phần số đầu    : 1000
  phần còn lại   : [100, 10, 2]
  nối bằng '/'   : 100/10/2
  ghép với ngoặc : 1000/(100/10/2) -> giá trị 200
=> 1000/(100/10/2)



## 9. Code nộp LeetCode

Đây là cách 4 (công thức) — ngắn nhất và `O(n)`. Khi nộp lên LeetCode, `List` đã được import sẵn; ở đây ta giữ `from typing import List` ở đầu notebook.

In [19]:
class Solution:
    def optimalDivision(self, nums: List[int]) -> str:
        n = len(nums)
        if n == 1:
            return str(nums[0])
        if n == 2:
            return f"{nums[0]}/{nums[1]}"
        return f"{nums[0]}/({'/'.join(map(str, nums[1:]))})"


sol = Solution()
for nums, expected in examples:
    got = sol.optimalDivision(nums)
    print(nums, "->", got)
    assert got == expected
print("Chạy ví dụ: OK")

[1000, 100, 10, 2] -> 1000/(100/10/2)
[2, 3, 4] -> 2/(3/4)
[7] -> 7
[7, 3] -> 7/3
Chạy ví dụ: OK


## 10. Vì sao đúng

**Khẳng định:** với `n ≥ 3`, đáp án duy nhất là `a0/(a1/a2/.../a_{n-1})`.

**Chứng minh (tóm tắt, dùng dấu mũ).**

1. Giá trị mọi biểu thức là `∏ a_i^{s_i}`, `s_i ∈ {+1, −1}` (tử `+1`, mẫu `−1`). Với phép chia `X / Y`, mũ trong `X` giữ nguyên, mũ trong `Y` **bị đổi dấu**.
2. `s0 = +1` luôn đúng: `a0` luôn ở nhánh trái của mọi phép chia chứa nó (không bao giờ bị lật).
3. `s1 = −1` luôn đúng: xét phép chia ngoài cùng `X / Y`. Nếu `X = a0` thì `a1` là số đầu của `Y`, mà `a1` có mũ `+1` trong `Y` (số đầu luôn `+1` theo mục 2) rồi bị lật thành `−1`. Nếu `X` có từ 2 số, `a1` nằm trong `X` và theo quy nạp trên `X` có mũ `−1` (không bị đổi dấu vì `X` là nhánh trái).
4. Các `s_i` còn lại tuỳ ý. Vì `a_i ≥ 2`, `a_i > 1/a_i`, nên giá trị lớn nhất là khi `s2 = ... = s_{n-1} = +1`; tức là `a0 · a2 · ... · a_{n-1} / a1`.
5. Biểu thức `a0 / (a1/a2/.../a_{n-1})` đạt đúng dấu mũ đó: trong ngoặc `a1` ở tử, `a2..a_{n-1}` ở mẫu; cả ngoặc là nhánh phải của phép chia ngoài nên mọi mũ trong ngoặc bị lật: `a1 → −1`, `a2.. → +1`.
6. **Duy nhất:** do `a_i > 1/a_i` chặt nên biểu thức tối ưu buộc phải có đúng dấu mũ `+ − + + ... +`. Dấu mũ này chỉ ứng với **một** cách đặt ngoặc không thừa. Thật vậy, xét phép chia ngoài cùng `X / Y`: nếu `X` có từ hai số trở lên thì số đầu của `Y` (là `a_k` với `k ≥ 2`) bị lật thành `−1`, trái với yêu cầu `+`; vậy `X = a0` và `Y = a1..a_{n-1}` phải có dấu `+ − − ... −` (mọi số sau số đầu đều `−`). Với `Y = X'/Y'`: nếu `Y'` có từ hai số trở lên thì số thứ hai của `Y'` có mũ `−1` trong `Y'` rồi bị lật thành `+1` trong `Y`, trái yêu cầu; vậy `Y'` là một số đơn, và `X'` lại phải thoả cùng dạng. Lặp lại ta được `Y` là chuỗi chia trái-sang-phải `a1/a2/.../a_{n-1}`.

Ta kiểm chứng bằng code: với nhiều input ngẫu nhiên, **số cách đặt ngoặc đạt giá trị lớn nhất đúng bằng 1** và cách đó trùng với công thức.

In [20]:
rng = random.Random(553)
checked = 0
for _ in range(400):
    n = rng.randint(3, 8)
    hi = rng.choice([4, 10, 1000])
    nums = [rng.randint(2, hi) for _ in range(n)]
    results = [(signs_value(nums, sig), expr) for expr, sig in bracketings(nums, 0, n - 1)]
    best = max(v for v, _ in results)
    winners = [expr for v, expr in results if v == best]
    assert len(winners) == 1, (nums, winners)            # tối ưu duy nhất
    assert winners[0] == optimal_formula(nums)
    checked += 1
print("đã kiểm chứng tính duy nhất và công thức trên", checked, "input ngẫu nhiên")

đã kiểm chứng tính duy nhất và công thức trên 400 input ngẫu nhiên


## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Vét cạn mọi cách đặt ngoặc | `O(C(n−1)·n)` ≈ mũ | mũ | `n ≤ 10` chạy được, dùng làm oracle |
| 2. DP theo đoạn (max/min) | `O(n³)` | `O(n²)` ô | không cần công thức, mở rộng được |
| 3. Thử một cặp ngoặc | `O(n³)` | `O(n)` | dựa vào giả thuyết "một cặp là đủ" |
| 4. Công thức | `O(n)` | `O(n)` | **cách nộp chuẩn** |

Với cách 4: duyệt `nums` một lần để `join` → `O(n)`; chuỗi trả về dài `O(n)`.

Vì `n ≤ 10`, mọi cách trừ cách vét cạn ở `n` lớn đều chạy trong chớp mắt trên LeetCode; điểm quan trọng của bài là **nhìn ra cấu trúc toán học** để có lời giải gọn và tổng quát cho mọi `n`.

## 12. Bộ test

Gồm: ví dụ đề, edge case (`n = 1, 2, 3`, số bằng nhau, số lớn nhất `1000`, `n = 10`), phản ví dụ cho cách làm sai, **hàm kiểm tra tính hợp lệ** `is_valid_answer` (đúng thứ tự số, không ngoặc thừa, giá trị bằng giá trị tối ưu của vét cạn), đối chiếu các cách trên nhiều input ngẫu nhiên có seed, và test lớn đúng ràng buộc.

In [21]:
METHODS = {
    "brute": optimal_brute,
    "dp": optimal_dp,
    "one_pair": optimal_one_pair,
    "formula": optimal_formula,
}

def best_value_brute(nums):
    return max(signs_value(nums, sig) for _, sig in bracketings(nums, 0, len(nums) - 1))

def is_valid_answer(nums, expr):
    """Hợp lệ nếu: parse được, đúng thứ tự số, không ngoặc thừa, giá trị bằng giá trị tối đa."""
    try:
        ast = parse(expr)
    except ValueError:
        return False
    if numbers_in_order(ast) != list(nums):
        return False
    if has_redundant(ast):
        return False
    return eval_ast(ast) == best_value_brute(nums)

def check_all(nums, expected):
    for name, fn in METHODS.items():
        got = fn(list(nums))
        assert got == expected, (name, nums, got, expected)
        assert is_valid_answer(nums, got), (name, nums, got)

for nums, expected in examples:
    check_all(nums, expected)

# edge case
check_all([2], "2")
check_all([1000], "1000")
check_all([2, 2], "2/2")
check_all([1000, 2], "1000/2")
check_all([2, 3, 4], "2/(3/4)")
check_all([2, 2, 2], "2/(2/2)")                       # các số bằng nhau
check_all([2, 2, 2, 2], "2/(2/2/2)")
check_all([3, 2, 1000], "3/(2/1000)")
check_all([1000, 1000, 1000, 1000, 1000], "1000/(1000/1000/1000/1000)")
check_all([24, 3, 4, 2], "24/(3/4/2)")
check_all(list(range(2, 12)), "2/(3/4/5/6/7/8/9/10/11)")   # n = 10
print("edge case: OK")

edge case: OK


**Kiểm tra chính bộ kiểm tra:** `is_valid_answer` phải **từ chối** các biểu thức sai: ngoặc thừa, sai thứ tự số, giá trị không tối ưu, chuỗi hỏng.

In [22]:
nums = [1000, 100, 10, 2]
assert is_valid_answer(nums, "1000/(100/10/2)")
assert not is_valid_answer(nums, "1000/((100/10)/2)")      # ngoặc thừa
assert not is_valid_answer(nums, "(1000/(100/10/2))")      # bọc cả biểu thức
assert not is_valid_answer(nums, "1000/(100/(10/2))")      # giá trị 50, không tối ưu
assert not is_valid_answer(nums, "1000/100/10/2")          # giá trị 0.5
assert not is_valid_answer(nums, "1000/(100/2/10)")        # sai thứ tự số
assert not is_valid_answer(nums, "1000/(100/10/2")         # thiếu ngoặc
assert not is_valid_answer([7, 3], "7/(3)")                # ngoặc thừa ở n = 2
assert is_valid_answer([7, 3], "7/3")
print("is_valid_answer: OK")

is_valid_answer: OK


In [23]:
# Đối chiếu 4 cách trên nhiều input ngẫu nhiên nhỏ (seed cố định)
rng = random.Random(2024)
count = 0
for _ in range(600):
    n = rng.randint(1, 8)
    hi = rng.choice([2, 3, 5, 20, 1000])
    nums = [rng.randint(2, hi) for _ in range(n)]
    results = {name: fn(list(nums)) for name, fn in METHODS.items()}
    assert len(set(results.values())) == 1, (nums, results)
    assert is_valid_answer(nums, results["formula"]), nums
    count += 1
print("đã đối chiếu", count, "input ngẫu nhiên n<=8: OK")

# n = 10 với số lớn: bỏ vét cạn ở nhiều input, chỉ so 3 cách còn lại; vét cạn vài input
fast = {k: METHODS[k] for k in ("dp", "one_pair", "formula")}
for _ in range(100):
    nums = [rng.randint(2, 1000) for _ in range(10)]
    results = {name: fn(list(nums)) for name, fn in fast.items()}
    assert len(set(results.values())) == 1, (nums, results)
for _ in range(3):
    nums = [rng.randint(2, 1000) for _ in range(10)]
    assert optimal_brute(nums) == optimal_formula(nums)
    assert is_valid_answer(nums, optimal_formula(nums))
print("n = 10: OK")

đã đối chiếu 600 input ngẫu nhiên n<=8: OK


n = 10: OK


In [24]:
# Thời gian ở ràng buộc lớn nhất (n = 10, số lên tới 1000)
nums = [1000, 999, 998, 997, 996, 995, 994, 993, 992, 991]
for name, fn in METHODS.items():
    t0 = time.perf_counter()
    res = fn(list(nums))
    print(f"{name:9} {(time.perf_counter() - t0) * 1000:8.2f} ms  ->", res)

brute       100.63 ms  -> 1000/(999/998/997/996/995/994/993/992/991)
dp            0.46 ms  -> 1000/(999/998/997/996/995/994/993/992/991)
one_pair      0.47 ms  -> 1000/(999/998/997/996/995/994/993/992/991)
formula       0.01 ms  -> 1000/(999/998/997/996/995/994/993/992/991)


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Thêm ngoặc ở trường hợp `n = 2`

**Code sai:**

```python
return f"{nums[0]}/({'/'.join(map(str, nums[1:]))})"      # luôn bọc ngoặc
```

**Vì sao sai:** với `[7, 3]` ta được `"7/(3)"`, dấu ngoặc đó **không đổi thứ tự tính** nên là ngoặc thừa, bị đề loại. Phải tách riêng `n == 2`.

In [25]:
def wrong_always_paren(nums):
    if len(nums) == 1:
        return str(nums[0])
    return f"{nums[0]}/({'/'.join(map(str, nums[1:]))})"       # SAI ở n = 2

print("sai :", wrong_always_paren([7, 3]), "| thừa ngoặc:", has_redundant(parse(wrong_always_paren([7, 3]))))
print("đúng:", optimal_formula([7, 3]))

sai : 7/(3) | thừa ngoặc: True
đúng: 7/3


### Lỗi 2: Quên trường hợp `n = 1`

**Code sai:**

```python
if n == 2:
    return f"{nums[0]}/{nums[1]}"
return f"{nums[0]}/({'/'.join(map(str, nums[1:]))})"       # n = 1 -> "7/()"
```

**Vì sao sai:** với `[7]`, `nums[1:]` là list rỗng, `join` ra chuỗi rỗng nên được `"7/()"` — không phải biểu thức hợp lệ. Phải trả `"7"`.

In [26]:
def wrong_no_n1(nums):
    n = len(nums)
    if n == 2:
        return f"{nums[0]}/{nums[1]}"
    return f"{nums[0]}/({'/'.join(map(str, nums[1:]))})"       # SAI ở n = 1

print("sai :", wrong_no_n1([7]))
print("đúng:", optimal_formula([7]))

sai : 7/()
đúng: 7


### Lỗi 3: Đặt ngoặc sai chỗ (bọc phần đầu thay vì phần sau)

**Code sai:**

```python
return f"({nums[0]}/{nums[1]})/" + "/".join(map(str, nums[2:]))      # bọc phần ĐẦU
```

**Vì sao sai:** ngoặc bọc **toán hạng trái** luôn là thừa (`(a/b)/c = a/b/c`) và cho giá trị **nhỏ nhất**, không phải lớn nhất. Ý muốn của ta là **đẩy các số sau `a1` lên tử**, nên ngoặc phải ở **bên phải dấu chia đầu tiên**, bao tất cả từ `a1` tới hết.

In [27]:
def wrong_paren_left(nums):
    if len(nums) <= 2:
        return "/".join(map(str, nums))
    return f"({nums[0]}/{nums[1]})/" + "/".join(map(str, nums[2:]))      # SAI

nums = [1000, 100, 10, 2]
bad = wrong_paren_left(nums)
print("sai :", bad, "=", evaluate(bad), "| thừa ngoặc:", has_redundant(parse(bad)))
good = optimal_formula(nums)
print("đúng:", good, "=", evaluate(good))

sai : (1000/100)/10/2 = 1/2 | thừa ngoặc: True
đúng: 1000/(100/10/2) = 200


### Lỗi 4: Quên `str()` khi nối chuỗi

**Code sai:**

```python
"/".join(nums[1:])         # nums là list số nguyên
```

**Vì sao sai:** `str.join` chỉ chấp nhận các **chuỗi**; gặp số nguyên sẽ báo `TypeError: sequence item 0: expected str instance, int found`. Hãy đổi bằng `map(str, ...)` hoặc list comprehension `[str(x) for x in ...]`.

In [28]:
nums = [1000, 100, 10, 2]
try:
    "/".join(nums[1:])
except TypeError as e:
    print("TypeError:", e)
print("đúng:", "/".join(map(str, nums[1:])))
print("đúng:", "/".join([str(x) for x in nums[1:]]))

TypeError: sequence item 0: expected str instance, int found
đúng: 100/10/2
đúng: 100/10/2


### Lỗi 5: So sánh giá trị bằng số thực, hoặc dùng `//`

**Code sai (khi tự viết bộ kiểm tra hay vét cạn):**

```python
value = a // b            # chia nguyên: làm tròn xuống, mất thông tin
value = a / b             # float: có sai số khi so sánh hai giá trị rất gần nhau
```

**Vì sao sai:** `1000//100//10//2` là `0` (mất hết thông tin so sánh), còn với `float` hai giá trị bằng nhau về toán học có thể lệch nhau ở chữ số cuối. Khi cần so sánh giá trị biểu thức, hãy dùng `Fraction` như trong notebook này.

In [29]:
print(1000 // 100 // 10 // 2, 1000 / 100 / 10 / 2, Fraction(1000, 100) / 10 / 2)
a, b = 0.1 + 0.2, 0.3
print("float :", a == b)
print("Fraction:", Fraction(1, 10) + Fraction(2, 10) == Fraction(3, 10))

0 0.5 1/2
float : False
Fraction: True


## 14. Mẹo & nhận dạng pattern

- **Bài "chọn dấu ngoặc để tối đa/tối thiểu biểu thức" chỉ có chia (hoặc chỉ có phép toán đơn giản):** hãy viết biểu thức dưới dạng **tích các số mũ `±1`** (hoặc tổng các số có dấu `±` nếu là trừ) rồi hỏi "dấu nào **bị ép cố định**, dấu nào **tự do**?". Cận trên/cận dưới thường đạt được bằng một cấu hình đơn giản.
- **Bài tương tự với dấu trừ:** `a0 − (a1 − a2 − ...)` — số đầu luôn `+`, số thứ hai luôn `−`, các số còn lại tự do.
- **Mẹo nghiệm:** vét cạn nhỏ (Catalan) trên vài input để **thấy mẫu** trước khi chứng minh; rồi dùng vét cạn làm oracle kiểm công thức.
- **Khi nào dùng DP max/min theo đoạn:** khi các số có thể âm/bằng 0/nhỏ hơn 1 hoặc có nhiều toán tử khác nhau (ví dụ *Different Ways to Add Parentheses*, *Minimum Score Triangulation*) và không có công thức gọn. Bài này đặc biệt vì `a_i ≥ 2`.
- **Về định dạng chuỗi:** đọc kỹ yêu cầu "không ngoặc thừa"; ngoặc ở toán hạng **trái** hay bọc **số đơn** luôn thừa.
- **Luôn tự viết một "máy tính nhỏ" (parser + `Fraction`)** để kiểm các đáp án dạng chuỗi biểu thức: nhanh và loại hẳn nhầm lẫn.

## 15. Biến thể

1. **Giá trị nhỏ nhất:** viết biểu thức cho giá trị **nhỏ** nhất. Vì `a0` luôn ở tử, `a1` luôn ở mẫu, các số khác tối ưu khi ở **mẫu** → biểu thức không ngoặc `a0/a1/a2/.../a_{n-1}`.
2. **Đếm số cách đặt ngoặc không thừa:** `C(n−1)` (số Catalan).
3. **Trả về giá trị tối đa (dạng phân số)** thay vì chuỗi: `a0·a2·...·a_{n−1} / a1`.
4. **Số có thể bằng 1 hoặc nhỏ hơn 1** (không còn `a_i ≥ 2`): công thức "một cặp ngoặc" **không còn đúng**; quay lại DP theo đoạn lưu `max/min` (cách 2).
5. **Thêm các toán tử khác** (`+`, `−`, `×`): *Different Ways to Add Parentheses* (241) — liệt kê hoặc DP đoạn.
6. **Phép trừ thay cho chia:** `a0 − a1 − a2 ...`, tối đa hoá bằng `a0 − (a1 − a2 − ... )` = `a0 − a1 + a2 + ... + a_{n−1}`.

Dưới đây là biến thể 1, 2 và 3 gọn trong một cell:

In [30]:
def min_division(nums):
    return "/".join(map(str, nums))                  # không ngoặc

def count_bracketings(n):
    m = n - 1                                        # Catalan(m) = C(2m, m)/(m+1)
    from math import comb
    return comb(2 * m, m) // (m + 1)

def max_value(nums):
    v = Fraction(nums[0])
    if len(nums) >= 2:
        v /= nums[1]
        for x in nums[2:]:
            v *= x
    return v

nums = [1000, 100, 10, 2]
print(min_division(nums), "=", evaluate(min_division(nums)))
print([count_bracketings(n) for n in range(1, 11)])
print(max_value(nums), evaluate(optimal_formula(nums)))
assert [count_bracketings(n) for n in range(1, 8)] == [len(bracketings(list(range(2, 2 + n)), 0, n - 1)) for n in range(1, 8)]
assert min_division(nums) == min((expr for expr, sig in bracketings(nums, 0, 3)), key=lambda e: evaluate(e))
assert max_value(nums) == evaluate(optimal_formula(nums))

1000/100/10/2 = 1/2
[1, 1, 2, 5, 14, 42, 132, 429, 1430, 4862]
200 200


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | Sinh mọi cách đặt ngoặc (số Catalan), chọn giá trị lớn nhất | `O(C(n−1)·n)` (mũ) | mũ | Chỉ để đối chiếu với `n` nhỏ |
| 2. DP theo đoạn | Mỗi đoạn lưu `max` và `min`; `max = max_trái / min_phải` | `O(n³)` | `O(n²)` ô | Bài tổng quát hơn (số bất kỳ, nhiều toán tử), không cần công thức |
| 3. Một cặp ngoặc | Chỉ thử các biểu thức có tối đa một cặp ngoặc | `O(n³)` | `O(n)` | Khi đã tin rằng "một cặp là đủ"; trung gian để dẫn tới công thức |
| 4. Công thức | `a0/(a1/a2/.../a_{n−1})` + xử lý riêng `n = 1, 2` | `O(n)` | `O(n)` | **Cách nộp chuẩn** cho bài này |

Kết luận: nộp **cách 4**. Cách 2 là công cụ dự phòng khi ràng buộc "số ≥ 2" bị bỏ. Cách 1 làm oracle.

## 17. Bài tập vận dụng (dễ → khó)

Mỗi bài có một cell để bạn làm. Các cell TODO bên dưới **chỉ khai báo** hàm rỗng hoặc gán `None`; hãy tự điền rồi bỏ comment dòng kiểm tra. Đáp án ở mục 19, gợi ý ở mục 18.

### Bài 1 (đoán output)

Đoán kết quả trước khi chạy (viết ra giấy), rồi chạy để so sánh.

In [31]:
print("/".join(map(str, [8, 4, 2])))
print(evaluate("8/(4/2)"))
print(evaluate("8/4/2"))
print(optimal_formula([8, 4, 2]))
print(optimal_formula([5]))
print(optimal_formula([5, 9]))

8/4/2
4
1
8/(4/2)
5
5/9


### Bài 2 (tính tay)

Với `nums = [24, 3, 4, 2]` có `5` cách đặt ngoặc không thừa. Hãy **liệt kê cả 5 chuỗi** cùng giá trị của chúng, gán vào `my_values` (dict từ chuỗi sang giá trị nguyên/phân số), và gán chuỗi tối ưu vào `my_best`. Hiện để `None`.

In [32]:
my_values = None      # ví dụ: {"24/3/4/2": 1, ...}
my_best = None
# Sau khi điền, bỏ comment để kiểm tra:
# truth = {e: signs_value([24, 3, 4, 2], s) for e, s in bracketings([24, 3, 4, 2], 0, 3)}
# assert my_values == truth and my_best == optimal_formula([24, 3, 4, 2])

### Bài 3 (điền TODO)

Hoàn thành `build_expression(nums)` bằng công thức (xử lý riêng `n == 1`, `n == 2`, còn lại bọc ngoặc quanh `nums[1:]`).

In [33]:
def build_expression(nums):
    n = len(nums)
    # TODO: n == 1
    # TODO: n == 2
    # TODO: n >= 3, dùng "/".join(map(str, ...)) và f-string
    pass

# Kiểm tra (bỏ comment sau khi viết xong):
# assert build_expression([7]) == "7"
# assert build_expression([7, 3]) == "7/3"
# assert build_expression([2, 3, 4]) == "2/(3/4)"
# assert build_expression([1000, 100, 10, 2]) == "1000/(100/10/2)"

### Bài 4 (tự viết lại)

Viết `max_value_formula(nums)` trả về **giá trị lớn nhất** (kiểu `Fraction`) **không dựng chuỗi**, chỉ bằng công thức `a0·a2·...·a_{n−1} / a1` (với `n = 1` là `a0`; với `n = 2` là `a0/a1`). Không xem lại code ở trên.

In [34]:
def max_value_formula(nums):
    # TODO
    pass

# assert max_value_formula([7]) == 7
# assert max_value_formula([7, 3]) == Fraction(7, 3)
# assert max_value_formula([2, 3, 4]) == Fraction(8, 3)
# assert max_value_formula([1000, 100, 10, 2]) == 200

### Bài 5 (biến thể / mini challenge)

Viết `min_division(nums)` trả về biểu thức có giá trị **nhỏ nhất** (không ngoặc thừa). Kiểm tra bằng cách so với vét cạn: giá trị của đáp án phải bằng **giá trị nhỏ nhất** trong mọi cách đặt ngoặc, và biểu thức phải không có ngoặc thừa. Gợi ý: hỏi "dấu nào bị ép cố định, dấu nào tự do" như ở mục 14.

In [35]:
def min_division(nums):
    # TODO
    return None

def valid_min(nums, expr):
    if expr is None:
        return False
    ast = parse(expr)
    smallest = min(signs_value(nums, s) for _, s in bracketings(nums, 0, len(nums) - 1))
    return numbers_in_order(ast) == list(nums) and not has_redundant(ast) and eval_ast(ast) == smallest

# for nums in ([7], [7, 3], [2, 3, 4], [1000, 100, 10, 2]):
#     assert valid_min(nums, min_division(nums)), nums

## 18. Gợi ý (3 mức)

**Bài 1.**
- Mức 1: `map(str, ...)` đổi số thành chuỗi; `"/".join(...)` nối bằng dấu `/`.
- Mức 2: `8/(4/2)` = `8·2/4`; `8/4/2` = `8/(4·2)`.
- Mức 3: kết quả theo thứ tự: `8/4/2`, `4`, `1`, `8/(4/2)`, `5`, `5/9`.

**Bài 2.**
- Mức 1: chia đoạn `[24, 3, 4, 2]` ở ba vị trí: sau `24`, sau `3`, sau `4`; mỗi vị trí cho vài cách con.
- Mức 2: các chuỗi: `24/3/4/2`, `24/(3/4)/2`, `24/3/(4/2)`, `24/(3/4/2)`, `24/(3/(4/2))`.
- Mức 3: giá trị lần lượt: `1`, `16`, `4`, `64`, `16`. Lớn nhất là `64` ứng với `24/(3/4/2)`.

**Bài 3.**
- Mức 1: `n == 1` trả `str(nums[0])`.
- Mức 2: `n == 2` trả `f"{nums[0]}/{nums[1]}"`, không ngoặc.
- Mức 3: `n >= 3`: `f"{nums[0]}/({'/'.join(map(str, nums[1:]))})"`.

**Bài 4.**
- Mức 1: dấu mũ tốt nhất là `+ − + + ... +`.
- Mức 2: bắt đầu `v = Fraction(nums[0])`, nếu `n ≥ 2` thì chia cho `nums[1]`.
- Mức 3: nhân tiếp với từng `x` trong `nums[2:]`.

**Bài 5.**
- Mức 1: số đầu luôn tử, số thứ hai luôn mẫu; các số còn lại tốt nhất ở đâu để biểu thức **nhỏ**?
- Mức 2: để mọi số còn lại ở **mẫu**, ta không đẩy số nào lên tử: không cần ngoặc.
- Mức 3: `"/".join(map(str, nums))`.

## 19. Đáp án tham khảo

Các đáp án dưới đây đều chạy kiểm tra bằng `assert`.

In [36]:
# Bài 1
assert "/".join(map(str, [8, 4, 2])) == "8/4/2"
assert evaluate("8/(4/2)") == 4
assert evaluate("8/4/2") == 1
assert optimal_formula([8, 4, 2]) == "8/(4/2)"
assert optimal_formula([5]) == "5"
assert optimal_formula([5, 9]) == "5/9"
print("Bài 1: 8/4/2, 4, 1, 8/(4/2), 5, 5/9")

# Bài 2
my_values = {
    "24/3/4/2": 1,
    "24/(3/4)/2": 16,
    "24/3/(4/2)": 4,
    "24/(3/4/2)": 64,
    "24/(3/(4/2))": 16,
}
my_best = "24/(3/4/2)"
truth = {e: signs_value([24, 3, 4, 2], s) for e, s in bracketings([24, 3, 4, 2], 0, 3)}
assert my_values == truth and my_best == optimal_formula([24, 3, 4, 2])
print("Bài 2: OK, tối ưu", my_best, "=", my_values[my_best])

Bài 1: 8/4/2, 4, 1, 8/(4/2), 5, 5/9
Bài 2: OK, tối ưu 24/(3/4/2) = 64


In [37]:
# Bài 3
def build_expression(nums):
    n = len(nums)
    if n == 1:
        return str(nums[0])
    if n == 2:
        return f"{nums[0]}/{nums[1]}"
    return f"{nums[0]}/({'/'.join(map(str, nums[1:]))})"

assert build_expression([7]) == "7"
assert build_expression([7, 3]) == "7/3"
assert build_expression([2, 3, 4]) == "2/(3/4)"
assert build_expression([1000, 100, 10, 2]) == "1000/(100/10/2)"
print("Bài 3: OK")

# Bài 4
def max_value_formula(nums):
    v = Fraction(nums[0])
    if len(nums) >= 2:
        v /= nums[1]
        for x in nums[2:]:
            v *= x
    return v

assert max_value_formula([7]) == 7
assert max_value_formula([7, 3]) == Fraction(7, 3)
assert max_value_formula([2, 3, 4]) == Fraction(8, 3)
assert max_value_formula([1000, 100, 10, 2]) == 200
rng = random.Random(1)
for _ in range(200):
    nums = [rng.randint(2, 1000) for _ in range(rng.randint(1, 8))]
    assert max_value_formula(nums) == best_value_brute(nums)
print("Bài 4: OK")

# Bài 5
def min_division(nums):
    return "/".join(map(str, nums))

for nums in ([7], [7, 3], [2, 3, 4], [1000, 100, 10, 2]):
    assert valid_min(nums, min_division(nums)), nums
rng = random.Random(2)
for _ in range(200):
    nums = [rng.randint(2, 1000) for _ in range(rng.randint(1, 8))]
    assert valid_min(nums, min_division(nums)), nums
print("Bài 5: OK")

Bài 3: OK


Bài 4: OK


Bài 5: OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template "chọn ngoặc cho biểu thức toàn phép chia":**

```python
class Solution:
    def optimalDivision(self, nums: List[int]) -> str:
        n = len(nums)
        if n == 1:
            return str(nums[0])
        if n == 2:
            return f"{nums[0]}/{nums[1]}"            # không ngoặc: ngoặc sẽ thừa
        return f"{nums[0]}/({'/'.join(map(str, nums[1:]))})"
```

**Ghi nhớ một câu:** *"Số đầu luôn ở tử, số thứ hai luôn ở mẫu; để lớn nhất, đẩy mọi số còn lại lên tử bằng một cặp ngoặc bao từ số thứ hai tới hết."*

**Checklist trước khi nộp:**

- [ ] Xử lý riêng `n == 1` (trả `str(nums[0])`) và `n == 2` (không ngoặc).
- [ ] Dùng `map(str, ...)` trước khi `join`.
- [ ] Ngoặc đặt **ngay sau dấu `/` đầu tiên** và đóng ở **cuối chuỗi**.
- [ ] Không có ngoặc bọc cả biểu thức, không bọc số đơn, không bọc toán hạng trái.
- [ ] Không có khoảng trắng trong chuỗi kết quả.
- [ ] Nếu tự viết kiểm tra: so sánh giá trị bằng `Fraction`, không bằng `float` hay `//`.
- [ ] Đã thử: `[7]`, `[7, 3]`, `[2, 3, 4]`, `[1000, 100, 10, 2]`, các số bằng nhau, `n = 10`.